# 10 — Phase 2: Residual Spatial Hybrid Architecture (AML + GNN Fusion)

## Motivation & Engineering Hypothesis
In our official IEEE report (Section VII: Conclusion and Phase 2 Outlook), we identified that:
1. **AML Tree Ensembles (Random Forest / XGBoost)** excel at localized segment trends by exploiting fine-grained temporal lags.
2. **Spatial Graph Neural Networks (GAT)** capture network topology and corridor connectivity, but risk over-smoothing extreme local peaks when trained standalone.

### The Residual Hybrid Hypothesis
Rather than choosing between AML and DL, we deploy a **two-stage hybrid architecture**:
* **Stage 1 (Base Predictor):** The localized AML model predicts the base speed $\hat{y}_{s,t}^{\text{AML}}$.
* **Residual Error Extraction:** We extract spatial errors: {s,t} = y_{s,t} - \hat{y}_{s,t}^{\text{AML}}$.
* **Stage 2 (Spatial Graph Compensator):** A PyTorch Graph Attention Network trains on the error residuals to learn propagating shockwaves.
* **Final Composite Prediction:** $\hat{y}_{s,t}^{\text{final}} = \hat{y}_{s,t}^{\text{AML}} + \hat{e}_{s,t}^{\text{GNN}}$.

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
import matplotlib.pyplot as plt

from traffic_prediction.data import load_speed_csv, prepare_speed_inputs, HORIZON_STEPS
from traffic_prediction.graph import build_distance_adjacency, compute_dual_diffusion_matrices
from traffic_prediction.hybrid import ResidualSpatialHybrid
from traffic_prediction.metrics import regression_metrics

print("PyTorch version:", torch.__version__)
print("Traffic Prediction components successfully imported.")

## 1. Load Road Adjacency & Compute Directed Dual Diffusion

In [ ]:
data_dir = Path("data")
dist_df = pd.read_csv(data_dir / "sensor_graph/distances_la_2012.csv")
with open(data_dir / "sensor_graph/graph_sensor_ids.txt") as f:
    sensor_ids = [line.strip() for line in f if line.strip()]

adj = build_distance_adjacency(dist_df, sensor_ids, threshold=0.1)
P_f, P_b = compute_dual_diffusion_matrices(adj)

print(f"Adjacency shape: {adj.shape}, Non-zero edges: {(adj > 0).sum()}")
print(f"Forward transition P_f shape: {P_f.shape}, Backward transition P_b shape: {P_b.shape}")

## 2. Load Speed Telemetry and Establish Chronological Split

In [ ]:
raw_df = load_speed_csv(data_dir / "METR-LA.csv")
speeds = prepare_speed_inputs(raw_df)
n_timestamps, n_sensors = speeds.shape
print(f"Loaded METR-LA: {n_timestamps} timestamps x {n_sensors} sensors")

train_end = int(n_timestamps * 0.70)
val_end = int(n_timestamps * 0.80)
print(f"Train: 0..{train_end}, Val: {train_end}..{val_end}, Test: {val_end}..{n_timestamps}")

## 3. Instantiate and Train Residual Spatial Hybrid Architecture

In [ ]:
hybrid_model = ResidualSpatialHybrid(
    adjacency=adj,
    input_size=10,
    hidden_size=32,
    output_size=4,
)
print("ResidualSpatialHybrid successfully instantiated with GAT compensator:")
print(hybrid_model.gat_compensator)

## 4. Verification of Residual Correction Integration
The composite output blends base localized speed with spatial GAT residual adjustments:
70554\hat{y}_{s,t} = \hat{y}_{s,t}^{\text{AML}} + \hat{e}_{s,t}^{\text{GAT}}70554